# Section 3 — Newton Divided Differences

[← LIP vs. Taylor](02_lip_vs_taylor.ipynb) ·
[Module 2](README.md) ·
[Next: Neville's Method →](04_neville_method.ipynb)

The Lagrange form is excellent for understanding interpolation, but the Newton
form is often more convenient computationally because new data can be added
without rebuilding the entire polynomial.

The Newton interpolating polynomial is

$$
P_n(x)=f[x_0]
+f[x_0,x_1](x-x_0)
+\cdots
+f[x_0,\ldots,x_n]
\prod_{j=0}^{n-1}(x-x_j).
$$

The coefficients are **divided differences**.

### Recursive definition

$$
f[x_i,x_{i+1}]
=
\frac{f[x_{i+1}]-f[x_i]}{x_{i+1}-x_i},
$$

and in general

$$
f[x_i,\ldots,x_{i+j}]
=
\frac{
f[x_{i+1},\ldots,x_{i+j}]
-
f[x_i,\ldots,x_{i+j-1}]
}{
x_{i+j}-x_i
}.
$$

In [1]:
import numpy as np

## Example based on the course divided-difference table

Suppose

$$
x_0=0.1,\qquad f[x_0]=2,
$$

$$
x_1=0.3,\qquad f[x_0,x_1]=20,
$$

and

$$
x_2=0.4,\qquad f[x_2]=11.
$$

First recover $f[x_1]$:

$$
20=\frac{f[x_1]-2}{0.3-0.1},
$$

so $f[x_1]=6$.

Then

$$
f[x_1,x_2]
=
\frac{11-6}{0.4-0.3}
=50,
$$

and

$$
f[x_0,x_1,x_2]
=
\frac{50-20}{0.4-0.1}
=100.
$$

In [2]:
x_nodes = np.array([0.1, 0.3, 0.4])
y_nodes = np.array([2.0, 6.0, 11.0])


def divided_difference_table(x_values, y_values):
    """Return the complete divided-difference table."""
    n = len(x_values)
    table = np.zeros((n, n))
    table[:, 0] = y_values

    for column in range(1, n):
        for row in range(n - column):
            numerator = table[row + 1, column - 1]
            numerator -= table[row, column - 1]
            denominator = x_values[row + column] - x_values[row]
            table[row, column] = numerator / denominator

    return table


table = divided_difference_table(x_nodes, y_nodes)
print(table)

[[  2.  20. 100.]
 [  6.  50.   0.]
 [ 11.   0.   0.]]


The Newton polynomial is therefore

$$
P_2(x)
=
2
+20(x-0.1)
+100(x-0.1)(x-0.3).
$$

In [3]:
def newton_value(x_nodes, coefficients, x_value):
    """Evaluate a Newton-form interpolation polynomial."""
    result = coefficients[0]
    product = 1.0

    for degree in range(1, len(coefficients)):
        product *= x_value - x_nodes[degree - 1]
        result += coefficients[degree] * product

    return result


coefficients = table[0, :]

for x_value in x_nodes:
    value = newton_value(x_nodes, coefficients, x_value)
    print(f"P({x_value:.1f}) = {value:.10f}")

P(0.1) = 2.0000000000
P(0.3) = 6.0000000000
P(0.4) = 11.0000000000


## Why divided differences are useful

If one new node $x_{n+1}$ is added, the existing coefficients remain unchanged.
Only the new highest-order divided difference must be computed. This makes the
Newton form naturally incremental.

### References

- Eric Sullivan, *Numerical Methods: An Inquiry-Based Approach with Python*.
- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 course lecture notes and review materials.


[Next: Neville's Method →](04_neville_method.ipynb)